<a href="https://colab.research.google.com/github/Tetsuya-Y/HCS2025-10-Yamada/blob/main/HCS2025_10_NewExperiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

2025年度10月HCS研究会静岡大学山田の発表内の新規実験のソースコード

In [ ]:
#ライブラリのインストール
!pip -q install --upgrade openai

import os, getpass, time, re, json
from typing import List, Dict, Tuple
from openai import OpenAI

#ChatGPTのAPIキーの入力
#実行するとChatGPTのAPIキーの入力が求められますので入力してください
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

client = OpenAI()

MODEL = "gpt-4o"          # 特徴抽出と会話の両方に使用
MAX_TURNS = 30            # 無限ループ防止(上限ターン数)
STOP_PHRASE = "これで議論を終えましょう．"


def file_to_data_url(filepath):
    import mimetypes, base64
    mime = mimetypes.guess_type(filepath)[0]
    with open(filepath, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

#タングラムの画像URL
#実行する際はソースコード内に画像URLを入力してください
IMAGES_A = [
    file_to_data_url(""),
    file_to_data_url(""),
    file_to_data_url(""),
    file_to_data_url(""),
    file_to_data_url(""),
    file_to_data_url(""),
]
IMAGES_B = [
    file_to_data_url(""),
    file_to_data_url(""),
    file_to_data_url(""),
    file_to_data_url(""),
    file_to_data_url(""),
    file_to_data_url(""),
]

#画像管理用のID付与
def make_ids(prefix: str, n: int) -> List[str]:
    return [f"{prefix}{i:03d}" for i in range(1, n+1)]
IDS_A = make_ids("A", len(IMAGES_A))
IDS_B = make_ids("B", len(IMAGES_B))

#ChatGPTによる事前認知
def extract_features_with_gpt(image_urls: List[str], ids: List[str], set_label: str) -> dict:
    instruction = (
        "出力は必ず1個のJSONのみ．説明文は不要．\n"
        "スキーマは**単一のオブジェクト**で次の通り：\n"
        "{\n"
        '  "set": string,\n'
        '  "images": [ { "id": string, "shape": string[], "abstract": string[] } ],\n'
        '  "feature": [ { "id": string[], "detail": string[] } ],\n'
        '  "strategy": string[]\n'
        "}\n"
        f"画像は与えられた順に次のIDを割り当ててください：{', '.join(ids)}．"
        "shape は図形や配置などの形状的特徴，abstract は比喩や連想などの抽象的特徴．"
        "feature には，対応関係や共通点を自然言語で detail に書き，関係する画像ID群を id に配列で示す．"
        "strategy はタングラム命名課題を効率化するための方針を3件．"
        "色名や表現は日本語の一般語で簡潔に，事実に基づく観察のみを記載．"
    )
    user_text = (
        f"あなたは画像特徴の要約者です．集合{set_label}の画像群を要約してください．"
        "スキーマには同一物の同定に役立つ具体的特徴（形状的特徴(どのような図形で構成されているか，図形ごとの配置の位置や接し方など)，抽象的特徴(何かに例えるとどのような表現ができるか）を可能な限り抽出してください．"
        "形状的特徴も抽象的特徴もタングラムごとに最低でも5つは抽出してください．"
        "全体像ではタングラムそれぞれの関係性や共通点を対応するタングラムのIDとともに自然言語で抽出する(id(対応するタングラムのID群),detail(関係性や共通点に基づく認知))．仮で3つ抽出する．"
        "以上のタングラムごとの特徴や全体像を踏まえてタングラム命名課題を効率的に行うための戦略を3つ考えてください．"
        "タングラム命名課題とは提示したタングラムそれぞれに共通の名前を付ける課題です．相手にも同じタングラムがそれぞれ見えているので相手と一緒に同じ名前を付けてください"
        "全体像と戦略はエージェントごとにそれぞれ抽出してください．"

    )
    content = [{"type":"text","text":user_text}] + [
        {"type":"image_url","image_url":{"url":u}} for u in image_urls
    ]
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":instruction},
            {"role":"user","content":content},
        ],
        temperature=0.2,
        max_tokens=1500,
        response_format={"type":"json_object"},
    )
    try:
        return json.loads(resp.choices[0].message.content)
    except Exception:
        return {
            "set": set_label,
            "images": [{"id": i, "shape" :[],"abstract" : []} for i in ids],
            "feature": [{"id":[],"detail":[]}for i in 3],
        }

features_A = extract_features_with_gpt(IMAGES_A, IDS_A, "A")
features_B = extract_features_with_gpt(IMAGES_B, IDS_B, "B")

def build_fact_sheet(feat: dict, title: str) -> str:
    lines = [f"【{title}の画像事実シート（非公開）】"]
    for item in feat.get("images", []):
        iid = item.get("id","?")
        shape = "，".join(item.get("shape", [])[:5]) or "-"
        ab = "，".join(item.get("abstract", [])[:5])
        lines.append(f"- {iid}：形状的特徴＝{shape}，抽象的特徴＝{ab}")
    for item in feat.get("feature", []):
        id = "，".join(item.get("id", [])[:3]) or "-"
        detail = "，".join(item.get("detail", [])[:3])
        lines.append(f"- {id}：{detail}")
    for item in feat.get("strategy", []):
        lines.append(f"- {item}")
    return "\n".join(lines)

FACT_SHEET_A = build_fact_sheet(features_A, "集合A")
FACT_SHEET_B = build_fact_sheet(features_B, "集合B")

#タングラム命名課題の説明・出力制限
CONDUCT_RULES = (
    "あなたは音声会話を想定して，自然な日本語だけで対話してください．"
    "タングラム命名課題を行ってください．"
    "タングラム命名課題とは提示したタングラムそれぞれに共通の名前を付ける課題です．相手にも同じタングラムがそれぞれ見えているので相手と一緒に同じ名前を付けてください"
    "お互いに同じタングラムを見て，そのタングラムに同じ名前をつけてください．"
    "お互いに見ている画像グループは同じなので，全てのタングラムに必ず対応するタングラムがあります．ただしIDなどの順番は異なる可能性があります．"
    "自分のタングラムはそれぞれ異なるタングラムです．"
    "相手の画像は見えていない前提です．"
    "同じタングラムなのか確証を持てない場合は確認のためにどれだけの対話をしても問題ありません．"
    "できる限り効率的にタングラム命名課題を進めてください．"
    "この課題における効率性は，少ない労力で課題を早く終わらせることです．"
    "効率的に進める方法は画像事実シート内のstorategyを基に，状況に応じて適切なコミュニケーション戦略をとってください．"
    "タングラムが同じ対象だと合意できたら，わかりやすい共通名を提案し，自然な一文で**両者のIDを口頭確認**してください：\n"
    "共通名の提案などでタングラムの共通名を出すときは『』で囲って出力してください．"
    "お互いが確認して初めて合意できたと判断してください．"
    "自分の画像にまだ命名ができていないタングラムがある場合は終了宣言をしないでください．"
    "すべての対応が決まったら，最後の文の末尾に必ず「" + STOP_PHRASE + "」を付けて締めてください．"
    "句読点は「，」「．」を使い，冗長な挨拶は避けてください．"
)

#内部状態の出力
FORMAT_SPEC = (
    "出力は必ず次のJSONだけを返してください．説明文や余計な文字は出さないでください．"
    '{"utterance":"自然な日本語の発話（句読点は，．）",'
    '"note":{"hypothesis":"短い仮説(見ているタングラムのIDを含める)",'
    '"evidence":["観察事実1","観察事実2"],'
    '"next_step":"次に行う行動"}}'
    " noteは観察可能な事実と方針の要約のみで，内部思考過程の逐語は書かないでください．"
)

SYSTEM_A = (
    "あなたはエージェントAです．以下はあなた専用の非公開メモです．相手と共有してはいけません．\n"
    + FACT_SHEET_A + "\n\n" + CONDUCT_RULES + "\n\n" + FORMAT_SPEC
)
SYSTEM_B = (
    "あなたはエージェントBです．以下はあなた専用の非公開メモです．相手と共有してはいけません．\n"
    + FACT_SHEET_B + "\n\n" + CONDUCT_RULES + "\n\n" + FORMAT_SPEC
)

agents = [
    {"name": "エージェントA", "system": SYSTEM_A},
    {"name": "エージェントB", "system": SYSTEM_B},
]

#合意発話から合意されたタングラムの確認
PAIR_PATTERNS = [
    r"『(?P<name>.+?)』として，私の(?P<A>A\d{3})とあなたの(?P<B>B\d{3})",
    r"『(?P<name>.+?)』で，私の(?P<A>A\d{3})とあなたの(?P<B>B\d{3})",
    r"(?P<A>A\d{3}).*?(?P<B>B\d{3}).*?『(?P<name>.+?)』として",
]
PAIR_RES = [re.compile(p) for p in PAIR_PATTERNS]

def collect_pair_from_text(text: str) -> List[Tuple[str,str,str]]:
    res = []
    for rx in PAIR_RES:
        m = rx.search(text)
        if m:
            res.append((m.group("A"), m.group("B"), m.group("name").strip()))
    return res


def chat_once_with_note(agent_idx: int, transcript_text: str, start=False) -> dict:
    me = agents[agent_idx]
    if start:
        user_text = (
            "同定課題を始めます．"
            "あなたから課題を進めるための対話を開始してください．"
        )
    else:
        user_text = (
            "議論を続けます．以下の会話ログを踏まえて，"
            "課題を進めるための適切な対話を行ってください．JSONのみで返してください．"
            "まだ対応しているか確認できていない画像があればその画像について同様に議論を進めてください．\n"
            "すべての画像に対して同定できるかの確認が終わった場合は終了合図を付けてください．\n"
            "----- 会話ログ -----\n" + (transcript_text or "(まだ無し)") + "\n---------------------"
        )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":me["system"]},
            {"role":"user","content":user_text},
        ],
        temperature=0.5,
        max_tokens=280,
        response_format={"type":"json_object"},
    )
    #トークン使用量の出力(過剰使用防止のため)
    usage = resp.usage if hasattr(resp, "usage") else None
    if usage:
        print(f"[{me['name']}] prompt={usage.prompt_tokens}, "
              f"completion={usage.completion_tokens}, total={usage.total_tokens}")

    try:
        data = json.loads(resp.choices[0].message.content)
        if "utterance" not in data: data["utterance"] = ""
        if "note" not in data or not isinstance(data["note"], dict):
            data["note"] = {"hypothesis":"", "evidence":[], "next_step":""}
        data["note"].setdefault("hypothesis","")
        data["note"].setdefault("evidence",[])
        data["note"].setdefault("next_step","")
        return data
    except Exception:
        return {
            "utterance": resp.choices[0].message.content.strip(),
            "note": {"hypothesis":"", "evidence":[], "next_step":""}
        }

#実行
transcript: List[str] = []
pairs: Dict[Tuple[str,str], str] = {}
notes: Dict[str, List[dict]] = {"エージェントA": [], "エージェントB": []}

#エージェントA開始
a0 = chat_once_with_note(0, "", start=True)
transcript.append(f"{agents[0]['name']}: {a0.get('utterance','')}")
notes["エージェントA"].append(a0.get("note", {}))
for A_id, B_id, nm in collect_pair_from_text(a0.get("utterance","")):
    pairs[(A_id, B_id)] = nm
print(transcript[-1])

if STOP_PHRASE not in a0.get("utterance",""):
    turn = 1
    while turn <= MAX_TURNS:
        b = chat_once_with_note(1, "\n".join(transcript))
        transcript.append(f"{agents[1]['name']}: {b.get('utterance','')}")
        notes["エージェントB"].append(b.get("note", {}))
        for A_id, B_id, nm in collect_pair_from_text(b.get("utterance","")):
            pairs[(A_id, B_id)] = nm
        print(transcript[-1])
        if STOP_PHRASE in b.get("utterance",""): break

        a = chat_once_with_note(0, "\n".join(transcript))
        transcript.append(f"{agents[0]['name']}: {a.get('utterance','')}")
        notes["エージェントA"].append(a.get("note", {}))
        for A_id, B_id, nm in collect_pair_from_text(a.get("utterance","")):
            pairs[(A_id, B_id)] = nm
        print(transcript[-1])
        if STOP_PHRASE in a.get("utterance",""): break

        turn += 1
        time.sleep(0.2)

print("\n=== 会話ログ（終了） ===")
print("\n".join(transcript))

if pairs:
    print("\n=== 合意された対応（共通名付き） ===")
    for (A_id, B_id), nm in pairs.items():
        print(f"- {A_id} ↔ {B_id} → 共通名：「{nm}」")
else:
    print("\n※ まだ合意された対応を抽出できませんでした．ログを確認してください．")

print("\n=== 内部状態のまとめ（各ターンの外部メモ） ===")
print("— エージェントA —")
print(json.dumps(notes["エージェントA"], ensure_ascii=False, indent=2))
print("\n— エージェントB —")
print(json.dumps(notes["エージェントB"], ensure_ascii=False, indent=2))

[エージェントA] prompt=1512, completion=126, total=1638
エージェントA: まず，私が見ているタングラムについてお話しします．大きな三角形が下部に配置されていて，何かが飛んでいるように見えるものがあります．このタングラムについて何か思い当たるものはありますか？
[エージェントB] prompt=1658, completion=120, total=1778
エージェントB: 私が見ている中で，大きな三角形が下部に配置され，飛行機のような形をしているものがあります．これが同じものであれば，両者のIDを確認しましょう．私の見ているものはB001です．
[エージェントA] prompt=1735, completion=97, total=1832
エージェントA: 私の見ているものはA001です．これが同じものであれば，名前を付けましょう．
[エージェントB] prompt=1762, completion=132, total=1894
エージェントB: これでB001とA001は同じタングラムと確認できましたので，このタングラムに『飛行機』という名前を付けましょう．次に，大きな三角形が左側に配置され，鳥が飛んでいるような形をしているタングラムについて話しましょう．
[エージェントA] prompt=1849, completion=173, total=2022
エージェントA: 私が見ている中で，大きな三角形が左側に配置され，鳥が羽ばたいているように見えるものがあります．これが同じものであれば，両者のIDを確認しましょう．私の見ているものはA003です．
[エージェントB] prompt=1913, completion=116, total=2029
エージェントB: 私の見ているものはB003です．これが同じものであれば，このタングラムに『鳥』という名前を付けましょう．
[エージェントA] prompt=1963, completion=136, total=2099
エージェントA: これでA003とB003は同じタングラムと確認できましたので，このタングラムに『鳥』という名前を付けましょう．次に，大きな三角形が中央に配置されていて，人が走っているように見えるタングラムについて話しましょう．
[エージェントB] p

In [ ]:
!pip freeze > requirements.txt